In [2]:
# Cell 1 — imports + config
import s3fs
import pandas as pd
import numpy as np

S3_OUT_ROOT = "s3://tradebot-config-tokyo/data/stageC/dataset=v1"
SYMBOL = "BTCUSDT"
SPLIT = "train"

fs = s3fs.S3FileSystem()
base = f"{S3_OUT_ROOT}/symbol={SYMBOL}/split={SPLIT}"

print("Base:", base)
print("Exists?", fs.exists(base))

Base: s3://tradebot-config-tokyo/data/stageC/dataset=v1/symbol=BTCUSDT/split=train
Exists? True


In [3]:
# Cell 2 — list files + read a sample (fast)
# On lit quelques fichiers pour valider rapidement.
paths = fs.glob(base.replace("s3://", "") + "/date=*/part-*.parquet")
paths = ["s3://" + p for p in sorted(paths)]
print("n_files:", len(paths))
print("first:", paths[0] if paths else None)

sample_paths = paths[:5]  # ajuste si tu veux plus
df = pd.concat([pd.read_parquet(p, engine="pyarrow") for p in sample_paths], ignore_index=True)

df.shape, df.columns[:20]

KeyboardInterrupt: 

In [1]:
# Cell 3 — schema / dtypes / missing ratio
display(df.dtypes)

missing = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
missing.head(30)

NameError: name 'df' is not defined

In [ ]:
# Cell 4 — required columns check (tu peux en ajouter)
required = [
    "id_symbol","id_t0","id_t1","id_date",
    "stgB_allow","label_C",
    "cfg_delta_sec","cfg_window_sec","cfg_horizon_sec","cfg_tp_bps","cfg_sl_bps",
    "fC_mid_ret_1s_bps","fC_mid_ret_5s_bps",
    "fC_spread_bps_at_t1","fC_quote_updates_20s",
    "fC_flow_signed_vol_20s","fC_flow_aggr_buy_ratio_20s",
]
missing_cols = [c for c in required if c not in df.columns]
missing_cols

In [ ]:
# Cell 5 — time sanity: t1 = t0 + delta
# (tolérance 0s car on est en epoch seconds)
t0 = pd.to_datetime(df["id_t0"], utc=True, errors="coerce")
t1 = pd.to_datetime(df["id_t1"], utc=True, errors="coerce")
delta = (t1.view("int64") - t0.view("int64")) // 1_000_000_000

print("delta_sec unique:", np.unique(delta[~pd.isna(delta)]).tolist()[:10])
print("expected:", int(df["cfg_delta_sec"].iloc[0]))
print("bad rows:", int((delta != int(df["cfg_delta_sec"].iloc[0])).sum()))

In [ ]:
# Cell 6 — label sanity
# label_C ∈ {-1,0,+1}
vc = df["label_C"].value_counts(dropna=False).sort_index()
vc

In [ ]:
# Cell 7 — exit fields sanity
# exit_t_sec should be -1 or >=1 and <= horizon (souvent <=120), sauf si NO_PRICE etc.
exit_t = pd.to_numeric(df["label_C_exit_t_sec"], errors="coerce")
print("exit_t min/max:", exit_t.min(), exit_t.max())

reasons = df["label_C_exit_reason"].value_counts(dropna=False)
reasons.head(20)

In [ ]:
# Cell 8 — feature sanity ranges (quick)
checks = {
    "fC_mid_ret_1s_bps": (-500, 500),
    "fC_mid_ret_5s_bps": (-2000, 2000),
    "fC_spread_bps_at_t1": (0, 50),            # BTCUSDT top spread très petit
    "fC_quote_updates_20s": (0, 10_000),       # large
    "fC_flow_aggr_buy_ratio_20s": (0, 1.0),    # ratio
}

for c, (lo, hi) in checks.items():
    if c not in df.columns:
        print("skip (missing):", c)
        continue
    x = pd.to_numeric(df[c], errors="coerce")
    bad = ((x < lo) | (x > hi)) & np.isfinite(x)
    print(f"{c}: finite={np.isfinite(x).mean():.3f}  bad={bad.mean():.6f}  p1/50/99={np.nanpercentile(x, [1,50,99])}")

In [ ]:
# Cell 9 — duplicates / keys sanity
# id_t0 + id_symbol devrait être unique dans un même split (souvent vrai).
dups = df.duplicated(subset=["id_symbol","id_t0"]).mean()
print("dup rate id_symbol+id_t0:", dups)

# id_date should match id_t0 date (UTC)
date_from_t0 = pd.to_datetime(df["id_t0"], utc=True).dt.strftime("%Y-%m-%d")
bad_date = (date_from_t0 != df["id_date"]).mean()
print("bad id_date vs id_t0 date:", bad_date)

In [4]:
# Cell 10
import s3fs
import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc

fs = s3fs.S3FileSystem()

# paths = ta liste de fichiers parquet StageC (déjà construite)
# ex: paths = sorted([f"s3://..." for ...])

dataset = ds.dataset(paths, format="parquet", filesystem=fs)

label_counts = {-1: 0, 0: 0, 1: 0}
nan_counts = {"fC_spread_bps_at_t1": 0, "fC_flow_signed_vol_20s": 0}
n_rows = 0

scanner = dataset.scanner(
    columns=["label_C", "fC_spread_bps_at_t1", "fC_flow_signed_vol_20s"],
    use_threads=True,
    batch_size=131072,   # tu peux monter à 262144 si RAM OK
)

for i, batch in enumerate(scanner.to_batches()):
    n = batch.num_rows
    n_rows += n

    lab = batch.column(0)  # label_C
    # counts per label
    for k in (-1, 0, 1):
        label_counts[k] += int(pc.sum(pc.equal(lab, pa.scalar(k, pa.int8()))).as_py())

    # NaNs (nulls)
    nan_counts["fC_spread_bps_at_t1"] += int(pc.sum(pc.is_null(batch.column(1))).as_py())
    nan_counts["fC_flow_signed_vol_20s"] += int(pc.sum(pc.is_null(batch.column(2))).as_py())

    if (i + 1) % 50 == 0:
        print(f"batches={i+1} rows={n_rows:,}")

nan_rates = {k: nan_counts[k] / max(n_rows, 1) for k in nan_counts}

print("rows:", n_rows)
print("label_counts:", label_counts)
print("nan_rates:", nan_rates)

batches=50 rows=5,141
batches=100 rows=15,442
batches=150 rows=22,150
batches=200 rows=25,951
batches=250 rows=36,012
batches=300 rows=39,085
batches=350 rows=45,067
batches=400 rows=50,229
batches=450 rows=60,046
batches=500 rows=61,456
rows: 61751
label_counts: {-1: 31146, 0: 1139, 1: 29466}
nan_rates: {'fC_spread_bps_at_t1': 0.0, 'fC_flow_signed_vol_20s': 0.00463150394325598}
